In [ ]:
# Task 1000 — Random Forest Modeling

This notebook develops the Random Forest forecasting pipeline for the multi-horizon asset price forecasting project.

The model predicts future returns rather than absolute future prices. Predicted returns are subsequently converted back into implied future prices.

The modeling framework covers:

- TSLA
- GOOGL
- BTCUSD
- ETHUSD

across seven forecast horizons:

- 30 Days
- 3 Months
- 6 Months
- 1 Year
- 3 Years
- 5 Years
- 10 Years

The workflow uses chronological train-test splitting and avoids random shuffling because financial observations are time-dependent.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
r2_score
)

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_MODELING = PROJECT_ROOT / "data" / "modeling"

RESULTS = PROJECT_ROOT / "results"
FORECASTS = RESULTS / "forecasts"
METRICS = RESULTS / "metrics"
MODELS = PROJECT_ROOT / "models"

In [3]:
FORECASTS.mkdir(parents=True, exist_ok=True)
METRICS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

In [4]:
tsla = pd.read_csv(DATA_MODELING / "TSLA_modeling.csv")
googl = pd.read_csv(DATA_MODELING / "GOOGL_modeling.csv")
btc = pd.read_csv(DATA_MODELING / "BTCUSD_modeling.csv")
eth = pd.read_csv(DATA_MODELING / "ETHUSD_modeling.csv")

In [7]:
tsla.head()

,Date,Open,High,Low,Close,Volume,Daily_Return,Rolling_Volatility_20D,Momentum_5D,Momentum_20D,...,Future_Close_3Y,Future_Return_3Y,Target_Date_5Y,Future_Date_5Y,Future_Close_5Y,Future_Return_5Y,Target_Date_10Y,Future_Date_10Y,Future_Close_10Y,Future_Return_10Y
0,2010-06-28,1.13333,1.13333,1.13333,1.13333,0,NaN,NaN,NaN,NaN,...,7.15733,5.315310,2015-06-28,2015-06-29,17.4680,14.412987,2020-06-28,2020-06-29,67.2900,58.373704
1,2010-06-29,1.26667,1.66667,1.16933,1.59267,281749140,0.405301,NaN,NaN,NaN,...,7.81200,3.904971,2015-06-29,2015-06-29,17.4680,9.967746,2020-06-29,2020-06-29,67.2900,41.249807
2,2010-06-30,1.71933,2.02800,1.55333,1.58867,257915910,-0.002512,NaN,NaN,NaN,...,7.81200,3.917321,2015-06-30,2015-06-30,17.8840,10.257215,2020-06-30,2020-06-30,71.9867,44.312557
3,2010-07-01,1.66667,1.72800,1.35133,1.46400,123447945,-0.078474,NaN,NaN,NaN,...,7.81200,4.336066,2015-07-01,2015-07-01,17.9433,11.256352,2020-07-01,2020-07-01,74.6433,49.985861
4,2010-07-02,1.53333,1.54000,1.24733,1.28000,77127105,-0.125683,NaN,NaN,NaN,...,7.85467,5.136461,2015-07-02,2015-07-02,18.6680,13.584375,2020-07-02,2020-07-02,80.5767,61.950547


In [8]:
# TSLA 30D RF modeling（非封装）

In [8]:
modeling_assets={'TSLA':tsla, 'GOOGL':googl,'BTCUSD':btc,'ETHUSD':eth}

In [9]:
feature_cols = [
    "Momentum_5D",
    "Momentum_20D",
    "Momentum_60D",
    "Price_to_MA20",
    "Price_to_MA60",
    "Rolling_Volatility_20D",
    "QQQ_Return",
    "Excess_Return_vs_QQQ"
]

target_col = "Future_Return_30D"
future_date_col='Future_Date_30D'

In [10]:
HORIZONS = [
    "30D",
    "3M",
    "6M",
    "1Y",
    "3Y",
    "5Y",
    "10Y"
]

In [11]:
for df in[tsla,googl, btc, eth]:
    df['Date']=pd.to_datetime(df['Date'])
    
    df[future_date_col] = pd.to_datetime(df[future_date_col])

In [12]:
for name, df in modeling_assets.items():
    print(name,df.shape)
    print(df['Date'].min(),df['Date'].max())
    print()

TSLA (4064, 46)
2010-06-28 00:00:00 2026-08-25 00:00:00

GOOGL (5539, 46)
2004-08-19 00:00:00 2026-08-25 00:00:00

BTCUSD (4168, 45)
2010-07-19 00:00:00 2026-08-26 00:00:00

ETHUSD (4037, 46)
2015-08-07 00:00:00 2026-08-25 00:00:00



In [10]:
df=modeling_assets['TSLA'].copy()

In [29]:
df=df.dropna(subset=feature_cols+[target_col,future_date_col]).copy()

df=df.sort_values('Date').reset_index(drop=True)

In [38]:
split_idx = int(len(df) * 0.8)

split_date = df.iloc[split_idx]["Date"]

train_df = df[
    (df["Date"] < split_date) &
    (df[future_date_col] < split_date)
].copy()

test_df = df[
    df["Date"] >= split_date
].copy()

In [39]:
print("Split Date:", split_date)

print(
    "Train:",
    train_df["Date"].min(),
    "to",
    train_df["Date"].max()
)

print(
    "Test:",
    test_df["Date"].min(),
    "to",
    test_df["Date"].max()
)

print(
    "Latest training future date:",
    train_df[future_date_col].max()
)

Split Date: 2024-06-09 00:00:00
Train: 2015-08-07 00:00:00 to 2024-05-09 00:00:00
Test: 2024-06-09 00:00:00 to 2026-08-25 00:00:00
Latest training future date: 2024-06-08 00:00:00


In [ ]:
### Random Forest Regression

Random Forest Regression is an ensemble machine learning model built from many decision trees. Each tree learns a set of rules from a random subset of the training data and a random subset of features. For a regression task, the final prediction is the average prediction across all trees.

In this project, the model uses financial features such as daily return, rolling volatility, momentum, and price-to-moving-average ratio to predict the future 30-day return. The advantage of Random Forest is that it can capture non-linear relationships between market features and future returns without requiring a pre-defined linear formula.

The model is trained using only the historical training period. The testing period is kept separate to evaluate whether the model can generalize to unseen market data.

In [55]:
# 建模需要的列
model_cols = feature_cols + [target_col]

# 删除训练集和测试集中的缺失值
train_df = train_df.dropna(subset=model_cols).copy()
test_df = test_df.dropna(subset=model_cols).copy()

# 重新生成 X 和 y
x_train = train_df[feature_cols]
y_train = train_df[target_col]

x_test = test_df[feature_cols]
y_test = test_df[target_col]

In [56]:
from sklearn import set_config
from sklearn.ensemble import RandomForestRegressor

set_config(display="text")

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=1
)

rf_model.fit(x_train, y_train)

print("Random Forest model fitted successfully.")


Random Forest model fitted successfully.


In [57]:
y_pred = rf_model.predict(x_test)


In [58]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

directional_accuracy = np.mean(
    np.sign(y_test.values) == np.sign(y_pred)
)

In [52]:
print("y_test NaN:", y_test.isna().sum())
print("y_pred NaN:", np.isnan(y_pred).sum())

print("y_test length:", len(y_test))
print("y_pred length:", len(y_pred))

y_test NaN: 30
y_pred NaN: 0
y_test length: 808
y_pred length: 808


In [59]:

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

directional_accuracy = np.mean(
    np.sign(y_test.values) == np.sign(y_pred)
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)
print("Directional Accuracy:", directional_accuracy)

MAE: 0.19200329363272284
RMSE: 0.23834129472963486
R2: -0.2317956179405607
Directional Accuracy: 0.5


In [65]:
baseline_pred = np.zeros(len(y_test))

baseline_mae = mean_absolute_error(
    y_test,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_pred
    )
)

In [66]:
metrics_df = pd.DataFrame({
    "Asset": ["TSLA"],
    "Model": ["Random Forest"],
    "Horizon": ["30D"],
    "Train_Samples": [len(train_df)],
    "Test_Samples": [len(test_df)],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "Directional_Accuracy": [directional_accuracy],
    "Baseline_MAE": [baseline_mae],
    "Baseline_RMSE": [baseline_rmse]
})

In [68]:
metrics_df.to_csv(
    METRICS / "TSLA_RF_30D_metrics.csv",
    index=False
)

In [62]:
predicted_price = (
    test_df["Close"].values
    * (1 + y_pred)
)
actual_price = test_df["Future_Close_30D"].values

In [63]:
prediction_df = pd.DataFrame({
    "Date": test_df["Date"].values,
    "Current_Close": test_df["Close"].values,
    "Future_Date_30D": test_df["Future_Date_30D"].values,
    "Actual_Return_30D": y_test.values,
    "Predicted_Return_30D": y_pred,
    "Actual_Future_Price": actual_price,
    "Predicted_Future_Price": predicted_price
})

In [64]:
prediction_df.to_csv(
    FORECASTS / "TSLA_RF_30D_predictions.csv",
    index=False
)

In [11]:
remaining_horizons = [
    "3M",
    "6M",
    "1Y",
    "3Y",
    "5Y",
    "10Y"
]

feature_cols = [
    "Daily_Return",
    "Momentum_5D",
    "Momentum_20D",
    "Momentum_60D",
    "Price_to_MA20",
    "Price_to_MA60",
    "Rolling_Volatility_20D",
    "QQQ_Return",
    "Excess_Return_vs_QQQ"
]

In [12]:
tsla_df = modeling_assets["TSLA"].copy()

In [13]:
required_cols = feature_cols.copy()

for horizon in remaining_horizons:
    required_cols += [
        f"Future_Return_{horizon}",
        f"Future_Date_{horizon}",
        f"Future_Close_{horizon}"
    ]

missing_cols = [
    col for col in required_cols
    if col not in tsla_df.columns
]

print("Missing columns:", missing_cols)

Missing columns: []


In [14]:
# TSLA 剩下6个horizons预测（封装））

In [15]:
feature_cols = [
    "Daily_Return",
    "Momentum_5D",
    "Momentum_20D",
    "Momentum_60D",
    "Price_to_MA20",
    "Price_to_MA60",
    "Rolling_Volatility_20D",
    "QQQ_Return",
    "Excess_Return_vs_QQQ"
]

In [6]:
def run_rf_horizon(
    df,
    asset_name,
    horizon,
    feature_cols,
    split_ratio=0.8
):
    # Create horizon-specific column names
    target_col = f"Future_Return_{horizon}"
    future_date_col = f"Future_Date_{horizon}"
    future_close_col = f"Future_Close_{horizon}"
    
    #  Copy data
    data = df.copy()

    # Make sure dates are datetime
    data["Date"] = pd.to_datetime(data["Date"])
    data[future_date_col] = pd.to_datetime(
        data[future_date_col]
    )

    # Keep only rows usable for this horizon
    data = data.dropna(
        subset=feature_cols + [
            target_col,
            future_date_col,
            future_close_col
        ]
    ).copy()

    data = data.sort_values(
        "Date"
    ).reset_index(drop=True)

    # Chronological split
    split_idx = int(
        len(data) * split_ratio
    )

    split_date = data.iloc[
        split_idx
    ]["Date"]

    #  Purge overlapping training targets
    train_df = data[
        (data["Date"] < split_date)
        &
        (data[future_date_col] < split_date)
    ].copy()

    test_df = data[
        data["Date"] >= split_date
    ].copy()

    # Basic validation
    if len(train_df) == 0:
        raise ValueError(
            f"{asset_name} {horizon}: empty training set"
        )

    if len(test_df) == 0:
        raise ValueError(
            f"{asset_name} {horizon}: empty test set"
        )

    # X and y
    X_train = train_df[feature_cols]
    y_train = train_df[target_col]

    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    # Train Random Forest
    rf_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(
        X_train,
        y_train
    )

    # Predict
    y_pred = rf_model.predict(
        X_test
    )

    # Evaluation
    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    r2 = r2_score(
        y_test,
        y_pred
    )

    directional_accuracy = np.mean(
        np.sign(y_test.values)
        ==
        np.sign(y_pred)
    )

    # Zero-return baseline
    baseline_pred = np.zeros(
        len(y_test)
    )

    baseline_mae = mean_absolute_error(
        y_test,
        baseline_pred
    )

    baseline_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            baseline_pred
        )
    )

    # Convert return prediction to price
    predicted_price = (
        test_df["Close"].values
        *
        (1 + y_pred)
    )

    actual_price = test_df[
        future_close_col
    ].values

    # Prediction output
    prediction_df = pd.DataFrame({
        "Date":
            test_df["Date"].values,

        "Current_Close":
            test_df["Close"].values,

        "Future_Date":
            test_df[future_date_col].values,

        "Actual_Return":
            y_test.values,

        "Predicted_Return":
            y_pred,

        "Actual_Future_Price":
            actual_price,

        "Predicted_Future_Price":
            predicted_price
    })

       # Metrics output - raw numeric values
    metrics = {
        "Asset": asset_name,
        "Model": "Random Forest",
        "Horizon": horizon,
        "Split_Date": split_date.strftime("%Y-%m-%d"),

        "Train_Samples": int(len(train_df)),
        "Test_Samples": int(len(test_df)),

        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
        "Directional_Accuracy": float(directional_accuracy),

        "Baseline_MAE": float(baseline_mae),
        "Baseline_RMSE": float(baseline_rmse),

        "MAE_Improvement_vs_Baseline": float(
            baseline_mae - mae
        ),

        "RMSE_Improvement_vs_Baseline": float(
            baseline_rmse - rmse
        )
    }
    
    # Report-friendly metrics
    metrics_display = {
        "Asset": asset_name,
        "Model": "Random Forest",
        "Horizon": horizon,
        "Split_Date": split_date.strftime("%Y-%m-%d"),

        "Train_Samples": int(len(train_df)),
        "Test_Samples": int(len(test_df)),

        "MAE_%": round(mae * 100, 2),
        "RMSE_%": round(rmse * 100, 2),

        "R2": round(r2, 3),

        "Directional_Accuracy_%": round(
            directional_accuracy * 100,
            2
        ),

        "Baseline_MAE_%": round(
            baseline_mae * 100,
            2
        ),

        "Baseline_RMSE_%": round(
            baseline_rmse * 100,
            2
        ),

        "MAE_Improvement_pp": round(
            (baseline_mae - mae) * 100,
            2
        ),

        "RMSE_Improvement_pp": round(
            (baseline_rmse - rmse) * 100,
            2
        )
    }

    # Technical checks
    checks = {
        "Latest_Train_Date":
            train_df["Date"].max(),

        "Latest_Train_Future_Date":
            train_df[future_date_col].max(),

        "Test_Start":
            test_df["Date"].min()
    }

    return (
        rf_model,
        prediction_df,
        metrics,
        metrics_display,
        checks
    )

In [43]:
model_30d,pred_30d,metrics_30d,metrics_display_30d,checks_3m=run_rf_horizon(modeling_assets['TSLA'],'TSLA','30D',feature_cols)

In [44]:
pred_30d.head()

,Date,Current_Close,Future_Date,Actual_Return,Predicted_Return,Actual_Future_Price,Predicted_Future_Price
0,2023-05-19,180.14,2023-06-20,0.523537,0.026463,274.45,184.907114
1,2023-05-22,188.87,2023-06-21,0.373749,0.034779,259.46,195.438624
2,2023-05-23,185.77,2023-06-22,0.424396,0.031486,264.61,191.619216
3,2023-05-24,182.90,2023-06-23,0.402952,0.026174,256.60,187.687260
4,2023-05-25,184.47,2023-06-26,0.306717,0.027365,241.05,189.517943


In [45]:
metrics_display_30d

{'Asset': 'TSLA',
 'Model': 'Random Forest',
 'Horizon': '30D',
 'Split_Date': '2023-05-19',
 'Train_Samples': 3163,
 'Test_Samples': 797,
 'MAE_%': 14.5,
 'RMSE_%': np.float64(19.05),
 'R2': -0.297,
 'Directional_Accuracy_%': np.float64(50.82),
 'Baseline_MAE_%': 12.54,
 'Baseline_RMSE_%': np.float64(16.91),
 'MAE_Improvement_pp': -1.96,
 'RMSE_Improvement_pp': np.float64(-2.14)}

In [33]:
model_3m,pred_3m,metrics_3m,metrics_display_3m,checks_3m=run_rf_horizon(modeling_assets['TSLA'],'TSLA','3M',feature_cols)

In [34]:
checks_3m

{'Latest_Train_Date': Timestamp('2022-12-30 00:00:00'),
 'Latest_Train_Future_Date': Timestamp('2023-03-30 00:00:00'),
 'Test_Start': Timestamp('2023-04-03 00:00:00')}

In [35]:
metrics_3m

{'Asset': 'TSLA',
 'Model': 'Random Forest',
 'Horizon': '3M',
 'Split_Date': '2023-04-03',
 'Train_Samples': 3090,
 'Test_Samples': 788,
 'MAE': 0.269381133021002,
 'RMSE': 0.3376835109137239,
 'R2': -0.3049791122451362,
 'Directional_Accuracy': 0.5241116751269036,
 'Baseline_MAE': 0.23244617325594524,
 'Baseline_RMSE': 0.3099765796810838,
 'MAE_Improvement_vs_Baseline': -0.03693495976505673,
 'RMSE_Improvement_vs_Baseline': -0.02770693123264012}

In [36]:
pred_3m.head()

,Date,Current_Close,Future_Date,Actual_Return,Predicted_Return,Actual_Future_Price,Predicted_Future_Price
0,2023-04-03,194.77,2023-07-03,0.436669,0.128333,279.82,219.765422
1,2023-04-04,192.58,2023-07-05,0.466819,0.038489,282.48,199.992201
2,2023-04-05,185.52,2023-07-05,0.522639,0.059619,282.48,196.580535
3,2023-04-06,185.06,2023-07-06,0.494326,0.084112,276.54,200.625798
4,2023-04-10,184.51,2023-07-10,0.461222,0.194429,269.61,220.384080


In [37]:
metrics_display_3m

{'Asset': 'TSLA',
 'Model': 'Random Forest',
 'Horizon': '3M',
 'Split_Date': '2023-04-03',
 'Train_Samples': 3090,
 'Test_Samples': 788,
 'MAE_%': 26.94,
 'RMSE_%': np.float64(33.77),
 'R2': -0.305,
 'Directional_Accuracy_%': np.float64(52.41),
 'Baseline_MAE_%': 23.24,
 'Baseline_RMSE_%': np.float64(31.0),
 'MAE_Improvement_pp': -3.69,
 'RMSE_Improvement_pp': np.float64(-2.77)}

In [39]:
pred_3m.to_csv(
    FORECASTS / "TSLA" / "TSLA_RF_3M_predictions.csv",
    index=False
)

In [50]:
remaining_horizons = [
    "6M",
    "1Y",
    "3Y",
    "5Y",
    "10Y"
]

In [52]:
tsla_metrics_raw = []
tsla_metrics_display = []

In [53]:
tsla_metrics_raw.extend([
    metrics_30d,
    metrics_3m
])

tsla_metrics_display.extend([
    metrics_display_30d,
    metrics_display_3m
])

In [56]:
for horizon in remaining_horizons:
    (
        rf_model,
        prediction_df,
        metrics,
        metrics_display,
        checks
    )=run_rf_horizon(modeling_assets['TSLA'],'TSLA',horizon,feature_cols)

    tsla_metrics_raw.append(metrics)

    tsla_metrics_display.append(metrics_display)

    prediction_df.to_csv(FORECASTS/'TSLA'/f'TSLA_RF_{horizon}_predictions.csv',index=False)

    print(f'{horizon} completed')

6M completed
1Y completed
3Y completed
5Y completed


ValueError: TSLA 10Y: empty training set

In [22]:
tsla_metrics_raw.append({
    "Asset": "TSLA",
    "Model": "Random Forest",
    "Horizon": "10Y",
    "Split_Date": np.nan,
    "Train_Samples": np.nan,
    "Test_Samples": np.nan,
    "MAE": np.nan,
    "RMSE": np.nan,
    "R2": np.nan,
    "Directional_Accuracy": np.nan,
    "Baseline_MAE": np.nan,
    "Baseline_RMSE": np.nan,
    "MAE_Improvement_vs_Baseline": np.nan,
    "RMSE_Improvement_vs_Baseline": np.nan,
    "Status": "Insufficient historical data"
})

NameError: name 'tsla_metrics_raw' is not defined

In [58]:
tsla_metrics_display.append({
    "Asset": "TSLA",
    "Model": "Random Forest",
    "Horizon": "10Y",
    "Split_Date": np.nan,
    "Train_Samples": np.nan,
    "Test_Samples": np.nan,
    "MAE_%": np.nan,
    "RMSE_%": np.nan,
    "R2": np.nan,
    "Directional_Accuracy_%": np.nan,
    "Baseline_MAE_%": np.nan,
    "Baseline_RMSE_%": np.nan,
    "MAE_Improvement_pp": np.nan,
    "RMSE_Improvement_pp": np.nan,
    "Status": "Insufficient historical data"
})

In [59]:
tsla_metrics_raw_df = pd.DataFrame(tsla_metrics_raw)
tsla_metrics_display_df = pd.DataFrame(tsla_metrics_display)

In [62]:
tsla_metrics_raw_df[
    tsla_metrics_raw_df["Horizon"] == "6M"
]

,Asset,Model,Horizon,Split_Date,Train_Samples,Test_Samples,MAE,RMSE,R2,Directional_Accuracy,Baseline_MAE,Baseline_RMSE,MAE_Improvement_vs_Baseline,RMSE_Improvement_vs_Baseline,Status
2,TSLA,Random Forest,6M,2023-01-23,2976.0,776.0,0.408017,0.503538,-0.746309,0.573454,0.315829,0.416222,-0.092188,-0.087317,NaN
3,TSLA,Random Forest,6M,2023-01-23,2976.0,776.0,0.408017,0.503538,-0.746309,0.573454,0.315829,0.416222,-0.092188,-0.087317,NaN
4,TSLA,Random Forest,6M,2023-01-23,2976.0,776.0,0.408017,0.503538,-0.746309,0.573454,0.315829,0.416222,-0.092188,-0.087317,NaN


In [63]:
tsla_metrics_raw_df = (
    tsla_metrics_raw_df
    .drop_duplicates(
        subset=["Asset", "Model", "Horizon"],
        keep="first"
    )
    .reset_index(drop=True)
)

In [65]:
tsla_metrics_raw_df

,Asset,Model,Horizon,Split_Date,Train_Samples,Test_Samples,MAE,RMSE,R2,Directional_Accuracy,Baseline_MAE,Baseline_RMSE,MAE_Improvement_vs_Baseline,RMSE_Improvement_vs_Baseline,Status
0,TSLA,Random Forest,30D,2023-05-19,3163.0,797.0,0.144986,0.190510,-0.297394,0.508156,0.125365,0.169102,-0.019621,-0.021408,NaN
1,TSLA,Random Forest,3M,2023-04-03,3090.0,788.0,0.269381,0.337684,-0.304979,0.524112,0.232446,0.309977,-0.036935,-0.027707,NaN
2,TSLA,Random Forest,6M,2023-01-23,2976.0,776.0,0.408017,0.503538,-0.746309,0.573454,0.315829,0.416222,-0.092188,-0.087317,NaN
3,TSLA,Random Forest,1Y,2022-08-26,2750.0,751.0,0.938558,1.209315,-10.146535,0.737683,0.364684,0.474777,-0.573873,-0.734539,NaN
4,TSLA,Random Forest,3Y,2021-01-26,1848.0,651.0,3.228182,3.696419,-26.928829,0.634409,0.540451,0.812051,-2.687731,-2.884368,NaN
5,TSLA,Random Forest,5Y,2019-06-21,942.0,550.0,4.654279,5.945491,-0.276984,1.000000,5.324335,7.485323,0.670056,1.539832,NaN
6,TSLA,Random Forest,10Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Insufficient historical data


In [66]:
tsla_metrics_display_df = (
    tsla_metrics_display_df
    .drop_duplicates(
        subset=["Asset", "Model", "Horizon"],
        keep="first"
    )
    .reset_index(drop=True)
)

In [67]:
tsla_metrics_display_df

,Asset,Model,Horizon,Split_Date,Train_Samples,Test_Samples,MAE_%,RMSE_%,R2,Directional_Accuracy_%,Baseline_MAE_%,Baseline_RMSE_%,MAE_Improvement_pp,RMSE_Improvement_pp,Status
0,TSLA,Random Forest,30D,2023-05-19,3163.0,797.0,14.50,19.05,-0.297,50.82,12.54,16.91,-1.96,-2.14,NaN
1,TSLA,Random Forest,3M,2023-04-03,3090.0,788.0,26.94,33.77,-0.305,52.41,23.24,31.00,-3.69,-2.77,NaN
2,TSLA,Random Forest,6M,2023-01-23,2976.0,776.0,40.80,50.35,-0.746,57.35,31.58,41.62,-9.22,-8.73,NaN
3,TSLA,Random Forest,1Y,2022-08-26,2750.0,751.0,93.86,120.93,-10.147,73.77,36.47,47.48,-57.39,-73.45,NaN
4,TSLA,Random Forest,3Y,2021-01-26,1848.0,651.0,322.82,369.64,-26.929,63.44,54.05,81.21,-268.77,-288.44,NaN
5,TSLA,Random Forest,5Y,2019-06-21,942.0,550.0,465.43,594.55,-0.277,100.00,532.43,748.53,67.01,153.98,NaN
6,TSLA,Random Forest,10Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Insufficient historical data


In [ ]:
### Note on the 10-Year Random Forest Horizon

The 10-year Random Forest model is not evaluated using the same out-of-sample framework as the shorter forecast horizons because the available TSLA history is insufficient for a leakage-safe chronological train-test split.

Constructing a 10-year future-return target requires each observation to have an additional ten years of future price data. As a result, only the earlier portion of the historical dataset contains valid 10-year targets.

After applying the chronological train-test split and removing training observations whose future target dates overlap with the test period, no valid training observations remain.

Therefore, the 10-year evaluation metrics are intentionally left blank rather than reporting results based on a method that would introduce look-ahead bias.

The 10-year horizon may still be considered later as a model-implied long-term scenario, but it should not be interpreted as a directly comparable out-of-sample forecast.

In [68]:
tsla_metrics_raw_df.to_csv(
    METRICS / "TSLA_RF_metrics_raw.csv",
    index=False
)

In [69]:
tsla_metrics_display_df.to_csv(
    METRICS / "TSLA_RF_metrics_display.csv",
    index=False
)

In [13]:
model_5y, pred_5y, metrics_5y, metrics_display_5y, checks_5y = run_rf_horizon(
    modeling_assets["TSLA"],
    "TSLA",
    "5Y",
    feature_cols
)

In [18]:
# GOOGL modeling

In [20]:
googl_df = modeling_assets["GOOGL"].copy()

required_cols = feature_cols.copy()

for horizon in [
    "30D", "3M", "6M", "1Y", "3Y", "5Y", "10Y"
]:
    required_cols += [
        f"Future_Return_{horizon}",
        f"Future_Date_{horizon}",
        f"Future_Close_{horizon}"
    ]

missing_cols = [
    col for col in required_cols
    if col not in googl_df.columns
]

print("Missing columns:", missing_cols)

Missing columns: []


In [21]:
googl_metrics_raw = []
googl_metrics_display = []

googl_horizons = [
    "30D",
    "3M",
    "6M",
    "1Y",
    "3Y",
    "5Y",
    "10Y"
]

(FORECASTS / "GOOGL").mkdir(
    parents=True,
    exist_ok=True
)

for horizon in googl_horizons:

    (
        model,
        prediction_df,
        metrics,
        metrics_display,
        checks
    ) = run_rf_horizon(
        modeling_assets["GOOGL"],
        "GOOGL",
        horizon,
        feature_cols
    )

    googl_metrics_raw.append(metrics)
    googl_metrics_display.append(metrics_display)

    prediction_df.to_csv(
        FORECASTS
        / "GOOGL"
        / f"GOOGL_RF_{horizon}_predictions.csv",
        index=False
    )

    print(f"GOOGL {horizon} completed.")

GOOGL 30D completed.
GOOGL 3M completed.
GOOGL 6M completed.
GOOGL 1Y completed.
GOOGL 3Y completed.
GOOGL 5Y completed.


ValueError: GOOGL 10Y: empty training set

In [23]:
googl_metrics_raw.append({
    "Asset": "GOOGL",
    "Model": "Random Forest",
    "Horizon": "10Y",
    "Split_Date": np.nan,
    "Train_Samples": np.nan,
    "Test_Samples": np.nan,
    "MAE": np.nan,
    "RMSE": np.nan,
    "R2": np.nan,
    "Directional_Accuracy": np.nan,
    "Baseline_MAE": np.nan,
    "Baseline_RMSE": np.nan,
    "MAE_Improvement_vs_Baseline": np.nan,
    "RMSE_Improvement_vs_Baseline": np.nan,
    "Status": "Insufficient historical data"
})

In [24]:
googl_metrics_display.append({
    "Asset": "GOOGL",
    "Model": "Random Forest",
    "Horizon": "10Y",
    "Split_Date": np.nan,
    "Train_Samples": np.nan,
    "Test_Samples": np.nan,
    "MAE_%": np.nan,
    "RMSE_%": np.nan,
    "R2": np.nan,
    "Directional_Accuracy_%": np.nan,
    "Baseline_MAE_%": np.nan,
    "Baseline_RMSE_%": np.nan,
    "MAE_Improvement_pp": np.nan,
    "RMSE_Improvement_pp": np.nan,
    "Status": "Insufficient historical data"
})

In [28]:
googl_metrics_raw_df = pd.DataFrame(
    googl_metrics_raw
)

googl_metrics_display_df = pd.DataFrame(
    googl_metrics_display
)

In [29]:
googl_metrics_display_df

,Asset,Model,Horizon,Split_Date,Train_Samples,Test_Samples,MAE_%,RMSE_%,R2,Directional_Accuracy_%,Baseline_MAE_%,Baseline_RMSE_%,MAE_Improvement_pp,RMSE_Improvement_pp,Status
0,GOOGL,Random Forest,30D,2022-03-17,4344.0,1092.0,7.25,9.13,-0.032,57.69,7.35,9.25,0.10,0.12,NaN
1,GOOGL,Random Forest,3M,2022-01-28,4269.0,1083.0,14.50,18.05,-0.159,58.26,14.69,18.24,0.19,0.18,NaN
2,GOOGL,Random Forest,6M,2021-11-17,4154.0,1071.0,21.46,28.24,-0.120,67.69,23.02,30.41,1.56,2.17,NaN
3,GOOGL,Random Forest,1Y,2021-06-25,3930.0,1046.0,34.24,43.19,-0.114,71.22,40.50,50.36,6.26,7.17,NaN
4,GOOGL,Random Forest,3Y,2019-11-21,3027.0,946.0,53.98,80.06,-0.181,100.00,87.98,114.76,34.00,34.69,NaN
5,GOOGL,Random Forest,5Y,2018-04-19,2121.0,845.0,53.91,66.98,-1.545,100.00,173.16,178.18,119.25,111.20,NaN
6,GOOGL,Random Forest,10Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Insufficient historical data


In [31]:
googl_metrics_raw_df.to_csv(
    METRICS / 'GOOGL_metrics'/"GOOGL_RF_metrics_raw.csv",
    index=False
)

In [32]:
googl_metrics_display_df.to_csv(
    METRICS / 'GOOGL_metrics'/"GOOGL_RF_metrics_display.csv",
    index=False
)

In [33]:
# BTCUSD modeling

In [34]:
crypto_feature_cols = [
    "Daily_Return",
    "Momentum_5D",
    "Momentum_20D",
    "Momentum_60D",
    "Price_to_MA20",
    "Price_to_MA60",
    "Rolling_Volatility_20D"
]

In [35]:
btc_df = modeling_assets["BTCUSD"].copy()

btc_missing = [
    col for col in crypto_feature_cols
    if col not in btc_df.columns
]

print("BTC missing:", btc_missing)

BTC missing: []


In [41]:
btc_metrics_raw = []
btc_metrics_display = []

btc_horizons = [
    "30D",
    "3M",
    "6M",
    "1Y",
    "3Y",
    "5Y",
    "10Y"
]

(FORECASTS / "BTCUSD").mkdir(
    parents=True,
    exist_ok=True
)

for horizon in btc_horizons:

    try:
        (
            model,
            prediction_df,
            metrics,
            metrics_display,
            checks
        ) = run_rf_horizon(
            modeling_assets["BTCUSD"],
            "BTCUSD",
            horizon,
            crypto_feature_cols
        )

        btc_metrics_raw.append(metrics)
        btc_metrics_display.append(metrics_display)

        prediction_df.to_csv(
            FORECASTS
            / "BTCUSD"
            / f"BTCUSD_RF_{horizon}_predictions.csv",
            index=False
        )

        print(f"BTCUSD {horizon} completed.")

    except ValueError as e:
        print(f"BTCUSD {horizon} skipped:", e)

BTCUSD 30D completed.
BTCUSD 3M completed.
BTCUSD 6M completed.
BTCUSD 1Y completed.
BTCUSD 3Y completed.
BTCUSD 5Y completed.
BTCUSD 10Y skipped: BTCUSD 10Y: empty training set


In [44]:
btc_metrics_raw.append({
    "Asset": "BTCUSD",
    "Model": "Random Forest",
    "Horizon": "10Y",
    "Split_Date": np.nan,
    "Train_Samples": np.nan,
    "Test_Samples": np.nan,
    "MAE": np.nan,
    "RMSE": np.nan,
    "R2": np.nan,
    "Directional_Accuracy": np.nan,
    "Baseline_MAE": np.nan,
    "Baseline_RMSE": np.nan,
    "MAE_Improvement_vs_Baseline": np.nan,
    "RMSE_Improvement_vs_Baseline": np.nan,
    "Status": "Insufficient historical data"
})

In [46]:
btc_metrics_display.append({
    "Asset": "BTCUSD",
    "Model": "Random Forest",
    "Horizon": "10Y",
    "Split_Date": np.nan,
    "Train_Samples": np.nan,
    "Test_Samples": np.nan,
    "MAE": np.nan,
    "RMSE": np.nan,
    "R2": np.nan,
    "Directional_Accuracy": np.nan,
    "Baseline_MAE": np.nan,
    "Baseline_RMSE": np.nan,
    "MAE_Improvement_vs_Baseline": np.nan,
    "RMSE_Improvement_vs_Baseline": np.nan,
    "Status": "Insufficient historical data"
})

In [47]:
btc_metrics_raw_df = pd.DataFrame(
    btc_metrics_raw
)

btc_metrics_display_df = pd.DataFrame(
    btc_metrics_display
)

In [48]:
btc_metrics_display_df

,Asset,Model,Horizon,Split_Date,Train_Samples,Test_Samples,MAE_%,RMSE_%,R2,Directional_Accuracy_%,...,MAE_Improvement_pp,RMSE_Improvement_pp,MAE,RMSE,Directional_Accuracy,Baseline_MAE,Baseline_RMSE,MAE_Improvement_vs_Baseline,RMSE_Improvement_vs_Baseline,Status
0,BTCUSD,Random Forest,30D,2023-05-29,3246.0,818.0,15.09,19.88,-0.810,49.51,...,-3.91,-4.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BTCUSD,Random Forest,3M,2023-04-10,3169.0,809.0,52.78,65.88,-4.504,54.64,...,-29.79,-36.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BTCUSD,Random Forest,6M,2023-01-27,3052.0,796.0,121.95,155.88,-11.337,68.34,...,-83.48,-105.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BTCUSD,Random Forest,1Y,2022-09-02,2819.0,770.0,465.83,736.21,-119.743,74.55,...,-384.75,-640.43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,BTCUSD,Random Forest,3Y,2021-01-29,1890.0,667.0,3130.53,9421.32,-3604.550,99.40,...,-2945.84,-9179.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,BTCUSD,Random Forest,5Y,2019-06-24,958.0,563.0,24190.84,30078.05,-4866.894,100.00,...,-23579.08,-29329.65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,BTCUSD,Random Forest,10Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Insufficient historical data


In [50]:
btc_metrics_raw_df.to_csv(
    METRICS / 'BTCUSD_metrics'/"BTCUSD_RF_metrics_raw.csv",
    index=False
)

In [51]:
btc_metrics_display_df.to_csv(
    METRICS / 'BTCUSD_metrics'/"BTCUSD_RF_metrics_display.csv",
    index=False
)

In [52]:
# ETHUSD modeling

In [53]:
eth_df = modeling_assets["ETHUSD"].copy()

eth_missing = [
    col for col in crypto_feature_cols
    if col not in eth_df.columns
]

print("ETH missing:", eth_missing)

ETH missing: []


In [54]:
eth_metrics_raw = []
eth_metrics_display = []

eth_horizons = [
    "30D",
    "3M",
    "6M",
    "1Y",
    "3Y",
    "5Y",
    "10Y"
]

(FORECASTS / "ETHUSD").mkdir(
    parents=True,
    exist_ok=True
)

for horizon in eth_horizons:

    try:
        (
            model,
            prediction_df,
            metrics,
            metrics_display,
            checks
        ) = run_rf_horizon(
            modeling_assets["ETHUSD"],
            "ETHUSD",
            horizon,
            crypto_feature_cols
        )

        eth_metrics_raw.append(metrics)
        eth_metrics_display.append(metrics_display)

        prediction_df.to_csv(
            FORECASTS
            / "ETHUSD"
            / f"ETHUSD_RF_{horizon}_predictions.csv",
            index=False
        )

        print(f"ETHUSD {horizon} completed.")

    except ValueError as e:
        print(f"ETHUSD {horizon} skipped:", e)

ETHUSD 30D completed.
ETHUSD 3M completed.
ETHUSD 6M completed.
ETHUSD 1Y completed.
ETHUSD 3Y completed.
ETHUSD 5Y skipped: ETHUSD 5Y: empty training set
ETHUSD 10Y skipped: ETHUSD 10Y: empty training set


In [55]:
for horizon in ["5Y", "10Y"]:
    eth_metrics_raw.append({
        "Asset": "ETHUSD",
        "Model": "Random Forest",
        "Horizon": horizon,
        "Split_Date": np.nan,
        "Train_Samples": np.nan,
        "Test_Samples": np.nan,
        "MAE": np.nan,
        "RMSE": np.nan,
        "R2": np.nan,
        "Directional_Accuracy": np.nan,
        "Baseline_MAE": np.nan,
        "Baseline_RMSE": np.nan,
        "MAE_Improvement_vs_Baseline": np.nan,
        "RMSE_Improvement_vs_Baseline": np.nan,
        "Status": "Insufficient historical data"
    })

In [56]:
for horizon in ["5Y", "10Y"]:
    eth_metrics_display.append({
        "Asset": "ETHUSD",
        "Model": "Random Forest",
        "Horizon": horizon,
        "Split_Date": np.nan,
        "Train_Samples": np.nan,
        "Test_Samples": np.nan,
        "MAE_%": np.nan,
        "RMSE_%": np.nan,
        "R2": np.nan,
        "Directional_Accuracy_%": np.nan,
        "Baseline_MAE_%": np.nan,
        "Baseline_RMSE_%": np.nan,
        "MAE_Improvement_pp": np.nan,
        "RMSE_Improvement_pp": np.nan,
        "Status": "Insufficient historical data"
    })

In [57]:
eth_metrics_raw_df = pd.DataFrame(
    eth_metrics_raw
)

eth_metrics_display_df = pd.DataFrame(
    eth_metrics_display
)

In [58]:
eth_metrics_display_df

,Asset,Model,Horizon,Split_Date,Train_Samples,Test_Samples,MAE_%,RMSE_%,R2,Directional_Accuracy_%,Baseline_MAE_%,Baseline_RMSE_%,MAE_Improvement_pp,RMSE_Improvement_pp,Status
0,ETHUSD,Random Forest,30D,2024-05-28,3127.0,790.0,20.22,24.94,-0.343,48.86,16.34,21.52,-3.88,-3.42,NaN
1,ETHUSD,Random Forest,3M,2024-04-09,3017.0,777.0,70.62,94.61,-4.402,41.70,33.27,40.71,-37.35,-53.90,NaN
2,ETHUSD,Random Forest,6M,2024-01-28,2852.0,760.0,220.07,296.88,-35.292,34.74,35.61,49.32,-184.46,-247.56,NaN
3,ETHUSD,Random Forest,1Y,2023-09-03,2524.0,723.0,965.39,1327.59,-977.662,47.99,37.09,43.15,-928.29,-1284.44,NaN
4,ETHUSD,Random Forest,3Y,2022-01-26,1208.0,577.0,2791.39,3938.87,-2431.607,76.43,74.87,102.43,-2716.52,-3836.45,NaN
5,ETHUSD,Random Forest,5Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Insufficient historical data
6,ETHUSD,Random Forest,10Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Insufficient historical data


In [59]:
eth_metrics_raw_df.to_csv(
    METRICS / 'ETHUSD_metrics'/"ETHUSD_RF_metrics_raw.csv",
    index=False
)

In [60]:
eth_metrics_display_df.to_csv(
    METRICS / 'ETHUSD_metrics'/"ETHUSD_RF_metrics_display.csv",
    index=False
)